In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
import os
import sys
import re
import time
import html
import warnings
from datetime import datetime
from typing import List, Tuple, Optional

import pandas as pd
import pyodbc
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

try:
    import winsound
except ImportError:
    winsound = None

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------- #
# --------- CONFIGURE --------- #
# ----------------------------- #

DB_STRUCTURE_PATH = r"C:\Users\IN10052060\Desktop\Python\SqlDBStructure.xlsx"
EXCEL_SHEET_NAME = "Medicare"  # Columns: A=Facility, B=Database, C=Server

SQL_FOLDER = r"C:\Users\IN10052060\Downloads\Bad Debt Query"
OUTPUT_DIR = r"C:\Users\IN10052060\Downloads\Bad Debt Data"

DEFAULT_TIMEOUT_SECONDS = 120   # Allow heavier batches to finish
ENABLE_ODBC_ENCRYPTION = False  # Set True if your env requires encryption

DB_PAUSE_SECONDS = 0.20         # Small pause between DBs to reduce tempdb races

# Optional: set True to log duplicate column names encountered in result sets
LOG_DUPLICATE_COLUMNS = False

# ----------------------------- #
# ---------- HELPERS ---------- #
# ----------------------------- #

def ensure_dir(path: str) -> None:
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def safe_strip(val) -> Optional[str]:
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s if s else None
    return str(val).strip()

def is_invalid_cell(text: Optional[str]) -> bool:
    if text is None:
        return True
    t = text.strip()
    if not t:
        return True
    if t.startswith("="):
        return True
    return False

def read_db_targets_from_excel(xlsx_path: str, sheet_name: str) -> pd.DataFrame:
    try:
        wb = load_workbook(xlsx_path, data_only=True)
    except Exception as e:
        raise RuntimeError(f"Failed to open DB structure file: {xlsx_path}\n{e}")

    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found in '{xlsx_path}'. Available: {wb.sheetnames}")

    ws = wb[sheet_name]

    targets: List[Tuple[Optional[str], str, str]] = []
    bad_rows: List[Tuple[int, Optional[str], Optional[str], Optional[str]]] = []

    # A=Facility, B=Database, C=Server
    for r in range(2, ws.max_row + 1):
        facility = safe_strip(ws.cell(row=r, column=1).value)
        database = safe_strip(ws.cell(row=r, column=2).value)
        server   = safe_strip(ws.cell(row=r, column=3).value)

        if is_invalid_cell(server) or is_invalid_cell(database):
            bad_rows.append((r, facility, database, server))
            continue

        targets.append((facility, database, server))

    if bad_rows:
        print("⚠️ Skipped rows with missing/formula values (row, facility, database, server):")
        for item in bad_rows[:10]:
            print("   ", item)
        if len(bad_rows) > 10:
            print(f"   ... and {len(bad_rows) - 10} more")

    if not targets:
        raise ValueError(
            "No valid Server/Database rows found.\n"
            "- Ensure formulas are evaluated (open & save in Excel with Ctrl+Alt+F9).\n"
            "- Or replace formulas with static values."
        )

    df = pd.DataFrame(targets, columns=["Facility", "Database", "Server"])
    df.drop_duplicates(inplace=True)
    df = df.dropna(subset=["Database", "Server"])
    return df

def read_sql_file(path: str) -> str:
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin-1"]
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def sanitize_query(sql_text: str) -> str:
    """
    - Removes standalone GO lines
    - Unescapes HTML entities (&gt;,&lt;,&amp;)
    - Converts named PK constraints on temp tables to unnamed PRIMARY KEY
      to avoid SQL 2714 in tempdb across concurrent sessions.
    - Injects SET NOCOUNT ON; and SET XACT_ABORT ON; at top if missing
    """
    # Remove 'GO' batch separators
    lines = []
    for line in sql_text.splitlines():
        if line.strip().upper() == "GO":
            continue
        lines.append(line)
    text = "\n".join(lines)

    # Unescape HTML entities (&amp;gt; -> >, etc.)
    text = html.unescape(text)

    # Replace named temp-table PK constraints like:
    #   CONSTRAINT PK_#act PRIMARY KEY ...
    # or any constraint whose name contains '#'
    # with an unnamed PRIMARY KEY
    # Handles optional [brackets] and case-insensitive
    text = re.sub(
        r"CONSTRAINT\s+\[?(?:PK_)?[^ \]]*#\w+\]?\s+PRIMARY\s+KEY",
        "PRIMARY KEY",
        text,
        flags=re.IGNORECASE
    )

    # Ensure SET NOCOUNT ON; is at top
    header = text.lstrip()
    if not header.upper().startswith("SET NOCOUNT ON;"):
        text = "SET NOCOUNT ON;\n" + text

    # Also add XACT_ABORT for cleaner failure behavior
    header2 = text.lstrip()
    if not header2.upper().startswith("SET XACT_ABORT ON;"):
        text = "SET XACT_ABORT ON;\n" + text

    return text

def clean_illegal_characters(value):
    if isinstance(value, str):
        return "".join(ch for ch in value if ch.isprintable())
    return value

# ----------------------------- #
#  MODIFIED: SAFER DF CLEANING  #
# ----------------------------- #

def _make_unique_columns(cols: List[object]) -> List[str]:
    """
    Coerce to strings, strip whitespace, and make names unique (A, A.1, A.2, ...).
    """
    str_cols = [str(c).strip() if c is not None else "" for c in cols]
    seen = {}
    unique_cols = []
    for c in str_cols:
        if c in seen:
            seen[c] += 1
            unique_cols.append(f"{c}.{seen[c]}")
        else:
            seen[c] = 0
            unique_cols.append(c)
    return unique_cols

def log_duplicate_columns(cols: List[object]) -> None:
    if not LOG_DUPLICATE_COLUMNS:
        return
    from collections import Counter
    # stringified (stripped) view to reflect how they will be processed
    str_cols = [str(c).strip() if c is not None else "" for c in cols]
    counter = Counter(str_cols)
    dups = [c for c, n in counter.items() if n > 1]
    if dups:
        print("ℹ️ Duplicate columns found and disambiguated:", ", ".join(dups))

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Defensive cleaner that avoids .dtype on DataFrame when duplicate column names exist.

    - Coerces all column labels to strings and strips whitespace.
    - Deduplicates column names by appending .1, .2, ... where needed.
    - Cleans only object (string-like) columns.
    - Drops duplicate rows.
    """
    if df is None or df.empty:
        return df

    df_cleaned = df.copy()

    # Log duplicates (optional) and enforce unique column names
    log_duplicate_columns(list(df_cleaned.columns))
    df_cleaned.columns = _make_unique_columns(list(df_cleaned.columns))

    # Clean only object dtype columns
    obj_cols = df_cleaned.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        df_cleaned[c] = df_cleaned[c].apply(clean_illegal_characters)

    # Final tidy-ups
    df_cleaned.columns = [col.strip() for col in df_cleaned.columns]
    df_cleaned.drop_duplicates(inplace=True)

    return df_cleaned

def pick_best_odbc_driver() -> str:
    try:
        drivers = pyodbc.drivers()
    except Exception:
        drivers = []
    for cand in ("ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server", "SQL Server"):
        if cand in drivers:
            return cand
    return "SQL Server"

# ----------------------------- #
#    CORE: ROBUST FETCH DATA    #
# ----------------------------- #

def fetch_data(
    database: str,
    server: str,
    query: str,
    timeout: int = DEFAULT_TIMEOUT_SECONDS,
    encrypt: bool = ENABLE_ODBC_ENCRYPTION
) -> Tuple[Optional[pd.DataFrame], int, float, str]:
    """
    Execute an entire multi-statement batch in ONE call.
    Advance through non-result sets until the first set with columns is found.
    Build DataFrame manually—no pandas.read_sql_query (which expects a result set immediately).
    """
    start = datetime.now()
    conn = None
    try:
        driver = pick_best_odbc_driver()
        conn_str = (
            f"Driver={{{driver}}};"
            f"Server={server};"
            f"Database={database};"
            "Trusted_Connection=Yes;"
            f"Timeout={timeout};"
        )
        if encrypt and ("ODBC Driver 17" in driver or "ODBC Driver 18" in driver):
            conn_str += "Encrypt=Yes;TrustServerCertificate=Yes;"

        print(f"Connecting to DB='{database}' on Server='{server}' using Driver='{driver}' ...")
        conn = pyodbc.connect(conn_str, timeout=timeout, autocommit=True)
        cur = conn.cursor()
        # Optional: cursor-level timeout (some environments require this)
        try:
            cur.timeout = timeout
        except Exception:
            pass

        # Execute entire script once (temp tables persist within this connection)
        cur.execute(query)

        # Walk through intermediate sets until we find the first result set with columns
        columns = None
        rows = None

        while True:
            if cur.description:
                columns = [c[0] for c in cur.description]
                rows = cur.fetchall()
                break
            if not cur.nextset():
                break  # No more sets; no result set produced

        duration = (datetime.now() - start).total_seconds()

        # No result set -> return empty DataFrame but Success
        if not columns:
            return (pd.DataFrame(), 0, duration, "Success")

        df = pd.DataFrame.from_records(rows, columns=columns)
        # MOD: Clean with safe function that handles duplicate column names
        df_cleaned = clean_dataframe(df)
        return (df_cleaned, len(df_cleaned), duration, "Success")

    except pyodbc.Error as e:
        duration = (datetime.now() - start).total_seconds()
        status = f"pyodbc.Error: {e}"
        print(f"❌ {status}")
        return (None, 0, duration, status)
    except Exception as e:
        duration = (datetime.now() - start).total_seconds()
        status = f"Error: {type(e).__name__}: {e}"
        print(f"❌ {status}")
        return (None, 0, duration, status)
    finally:
        if conn is not None:
            try:
                conn.close()
            except Exception:
                pass

def beep_success():
    try:
        if winsound:
            winsound.MessageBeep()
    except Exception:
        pass

# ----------------------------- #
# ------------ MAIN ----------- #
# ----------------------------- #

def main():
    # Validate inputs
    if not os.path.exists(DB_STRUCTURE_PATH):
        print(f"❌ DB_STRUCTURE_PATH not found: {DB_STRUCTURE_PATH}")
        sys.exit(1)
    if not os.path.exists(SQL_FOLDER):
        print(f"❌ SQL_FOLDER not found: {SQL_FOLDER}")
        sys.exit(1)
    ensure_dir(OUTPUT_DIR)

    # Load targets
    print("📘 Loading DB targets from Excel (evaluated values)...")
    df_targets = read_db_targets_from_excel(DB_STRUCTURE_PATH, EXCEL_SHEET_NAME)
    if df_targets.empty:
        print("❌ No valid targets after filtering.")
        sys.exit(1)

    servers_grouped = df_targets.groupby("Server", dropna=True)

    # Collect SQL files
    sql_files = [f for f in os.listdir(SQL_FOLDER) if f.lower().endswith((".sql", ".txt"))]
    if not sql_files:
        print(f"❌ No .sql or .txt files found in: {SQL_FOLDER}")
        sys.exit(1)

    # Process each SQL file
    for sql_file in sql_files:
        sql_path = os.path.join(SQL_FOLDER, sql_file)
        print(f"\n🔍 Running query from: {sql_file}")

        raw_query = read_sql_file(sql_path)
        query = sanitize_query(raw_query)

        # --- Overall counters across all servers for this SQL file ---
        overall_total = 0
        overall_succeeded = 0
        overall_failed = 0
        overall_failed_list: List[str] = []  # e.g., "TranMTTN@AHS-TRAN15.accretivehealth.local"

        for server, group in servers_grouped:
            print(f"\n➡️ Processing Server: {server}")

            # Prepare Excel workbook for this server
            master_file = Workbook()
            log_sheet = master_file.active
            log_sheet.title = "Log"
            raw_data_sheet = master_file.create_sheet("RawData")
            summary_sheet = master_file.create_sheet("Summary")

            log_sheet.append(["Server", "Database", "Status", "Row Count", "Duration (s)", "Timestamp"])
            summary_sheet.append(["Server", "Total Databases", "Succeeded", "Failed", "Failed DB Names"])

            dfs: List[pd.DataFrame] = []
            header_written = False

            # Per-server stats
            total_dbs = 0
            succeeded = 0
            failed = 0
            failed_db_names: List[str] = []

            # Loop through databases for this server
            for _, row in group.iterrows():
                database = row["Database"]
                total_dbs += 1
                overall_total += 1  # overall attempt

                start_time = datetime.now()

                df, row_count, duration, status = fetch_data(
                    database=database,
                    server=server,
                    query=query,
                    timeout=DEFAULT_TIMEOUT_SECONDS,
                    encrypt=ENABLE_ODBC_ENCRYPTION
                )

                # Log per-database result
                log_sheet.append([
                    server,
                    database,
                    status,
                    row_count,
                    round(duration, 2),
                    start_time.strftime("%Y-%m-%d %H:%M:%S")
                ])

                # Tally
                if status == "Success":
                    succeeded += 1
                    overall_succeeded += 1
                    if df is not None and not df.empty:
                        if not header_written:
                            for r in dataframe_to_rows(df, index=False, header=True):
                                raw_data_sheet.append(r)
                            header_written = True
                        else:
                            for r in dataframe_to_rows(df, index=False, header=False):
                                raw_data_sheet.append(r)
                        dfs.append(df)
                else:
                    failed += 1
                    overall_failed += 1
                    failed_db_names.append(str(database))
                    overall_failed_list.append(f"{database}@{server}")

                # Small pause to reduce tempdb constraint races
                if DB_PAUSE_SECONDS > 0:
                    time.sleep(DB_PAUSE_SECONDS)

            # Add per-server summary row
            summary_sheet.append([
                server,
                total_dbs,
                succeeded,
                failed,
                ", ".join(failed_db_names) if failed_db_names else ""
            ])

            # Console summary for the server
            if failed > 0:
                print(f"⚠️ Server '{server}': {succeeded}/{total_dbs} databases succeeded.")
                print(f"   Failed: {', '.join(failed_db_names)}")
            else:
                print(f"✅ Server '{server}': All {total_dbs} databases succeeded.")

            # Save result for this server
            timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
            base_name = os.path.splitext(sql_file)[0]
            safe_server = server.replace("\\", "_").replace("/", "_").replace(":", "_")
            output_file = os.path.join(OUTPUT_DIR, f"{base_name}_{safe_server}_{timestamp}.xlsx")

            master_file.save(output_file)

            # Combined shape info (best-effort)
            combined_shape = (0, 0)
            if dfs:
                try:
                    combined_shape = pd.concat(dfs, ignore_index=True).shape
                except Exception:
                    combined_shape = (sum(len(d) for d in dfs), len(dfs[0].columns))

            print(f"✅ Saved result for Server {server}: {output_file}")
            print(f"📊 Combined shape for {server}: {combined_shape}\n")

        # --- Overall summary for this SQL file ---
        print(f"🧮 Overall for {sql_file}: {overall_succeeded}/{overall_total} databases succeeded; {overall_failed} failed")
        if overall_failed_list:
            print("   ❌ Failed DBs: " + ", ".join(overall_failed_list))
        print("")  # spacer

    beep_success()
    print("🎉 All done.")

if __name__ == "__main__":
    main()


📘 Loading DB targets from Excel (evaluated values)...
⚠️ Skipped rows with missing/formula values (row, facility, database, server):
    (86, 'SSIN', None, None)

🔍 Running query from: bad debt tag code change.txt

➡️ Processing Server: AHS-TRAN10.accretivehealth.local
Connecting to DB='TranASIN' on Server='AHS-TRAN10.accretivehealth.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranCLIN' on Server='AHS-TRAN10.accretivehealth.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranECIN' on Server='AHS-TRAN10.accretivehealth.local' using Driver='ODBC Driver 17 for SQL Server' ...
✅ Server 'AHS-TRAN10.accretivehealth.local': All 3 databases succeeded.
✅ Saved result for Server AHS-TRAN10.accretivehealth.local: C:\Users\IN10052060\Downloads\Bad Debt Data\bad debt tag code change_AHS-TRAN10.accretivehealth.local_20260608-230104.xlsx
📊 Combined shape for AHS-TRAN10.accretivehealth.local: (2305, 15)


➡️ Processing Server: AHS-TRAN11.accretiv

In [3]:
import os
import pandas as pd

# --- CONFIGURATION ---
EXCEL_FOLDER = r"C:\Users\IN10052060\Downloads\Bad debt Data" # Update to your folder path
combined_df = pd.DataFrame()

# --- LOOP THROUGH EXCEL FILES ---
excel_files = [f for f in os.listdir(EXCEL_FOLDER) if f.endswith(".xlsx")]

print(f"Found {len(excel_files)} Excel files:")
for file in excel_files:
    file_path = os.path.join(EXCEL_FOLDER, file)
    try:
        # Read the 'RawData' sheet (adjust if needed)
        df = pd.read_excel(file_path, sheet_name="RawData")
        df["SourceFile"] = file  # Optional: track origin
        combined_df = pd.concat([combined_df, df], ignore_index=True)
        print(f"✅ Loaded: {file} ({df.shape[0]} rows)")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

# --- PREVIEW RESULT ---
print(f"\n📊 Combined shape: {combined_df.shape}")
print(combined_df.head())


Found 21 Excel files:
✅ Loaded: bad debt tag code change_AHS-TRAN10.accretivehealth.local_20260608-230104.xlsx (2305 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN11.accretivehealth.local_20260608-230108.xlsx (774 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN12.accretivehealth.local_20260608-230119.xlsx (4673 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN14.accretivehealth.local_20260608-230157.xlsx (0 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN15.accretivehealth.local_20260608-230213.xlsx (12272 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN16.accretivehealth.local_20260608-230358.xlsx (24427 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN18.accretivehealth.local_20260608-230430.xlsx (21840 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN19.accretivehealth.local_20260608-230448.xlsx (7417 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN20.accretivehealth.local_20260608-230541.xlsx (35324 rows)
✅ Loaded: bad debt tag code change_AHS-TRAN40.accretivehealth.local_2

In [4]:
combined_df.shape

(343547, 16)

In [5]:
combined_df.to_excel(os.path.join(EXCEL_FOLDER, "Combined_Output.xlsx"), index=False)


In [ ]:

z